In [3]:
pip install ipywidgets voila pyarrow matplotlib

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 29.6 MB/s  0:00:00
   ---------------------------------------- 0.0/4.6 MB ? eta -:--:--
   ---------------------------------------- 4.6/4.6 MB 25.3 MB/s  0:00:00

   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   ------------------------ --------------- 3/5 [ipywidgets]
   ------------------------ --------------- 3/5 [ipywidgets]
   ------------------------ --------------- 3/5 [ipywidgets]
   ------------------------ --------------- 3/5 [ipywidgets]
   ------------------------ --------------- 3/5 [ipywidgets]
   -------------------------------- ------

In [6]:
# Libraries and packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets


In [2]:
# Pathing the project directories
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent
]

project_root = next(
    (
        root
        for root in candidate_roots
        if (
            root
            / 'data' / 'processed' / 'ca1_model_features.parquet'
        ).exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        'Project root not found.'
    )

features_path = (
    project_root
    / 'data' / 'processed' / 'ca1_model_features.parquet'
)

predictions_path = (
    project_root
    / 'data' / 'processed' / 'ca1_final_test_predictions.parquet'
)

metrics_path = (
    project_root
    / 'reports' / 'final_test_metrics.csv'
)

shap_path = (
    project_root
    / 'reports' / 'random_forest_grouped_shap.csv'
)

importance_path = (
    project_root
    / 'reports' / 'random_forest_grouped_importance.csv'
)

print(f'Project root: {project_root}')

Project root: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting


In [3]:
# Loading the stored data for the app
feature_df = pd.read_parquet(features_path)

all_predictions = pd.read_parquet(predictions_path)

metrics_df = pd.read_csv(metrics_path)

feature_df['date'] = pd.to_datetime(feature_df['date'])

all_predictions['date'] = pd.to_datetime(all_predictions['date'])

prediction_df = all_predictions.loc[
    all_predictions['model']
    == 'random_forest_tuned'
].copy()

if shap_path.exists():
    importance_df = pd.read_csv(shap_path)
    importance_value = ('mean_absolute_shap')
    importance_label = ('Mean absolute SHAP value')

elif importance_path.exists():
    importance_df = pd.read_csv(importance_path)
    importance_value = 'importance'

    importance_label = ('Random Forest importance')

else:
    importance_df = pd.DataFrame()
    importance_value = None
    importance_label = None

item_lookup = (
    feature_df[
        [ 'store_id', 'item_id', 'dept_id', 'cat_id']
    ]
    .drop_duplicates()
    .sort_values('item_id')
    .reset_index(drop=True)
)

print(
    f'Supported products: '
    f'{item_lookup["item_id"].nunique()}'
)

print(
    f'Random Forest forecast rows: '
    f'{len(prediction_df):,}'
)

Supported products: 108
Random Forest forecast rows: 3,024


### Stock planning calculation function

In [4]:
def calculate_stock_plan(forecast_total, current_stock, safety_percentage):
    if forecast_total < 0:
        raise ValueError('Forecast demand cannot be negative.')

    if current_stock < 0:
        raise ValueError('Current stock cannot be negative.')

    if not 0 <= safety_percentage <= 50:
        raise ValueError('Safety percentage must be ', 'between 0 and 50.')

    base_requirement = int(np.ceil(forecast_total))

    safety_stock = int(np.ceil(
            forecast_total
            * safety_percentage
            / 100
    ))

    suggested_stock = ( base_requirement + safety_stock )

    stock_gap = max( suggested_stock - current_stock, 0 )

    expected_surplus = max( current_stock - suggested_stock, 0 )

    return {
            'base_requirement':
                base_requirement,
            'safety_stock':
                safety_stock,
            'suggested_stock':
                suggested_stock,
            'stock_gap':
                stock_gap,
            'expected_surplus':
                expected_surplus
        }

    

In [7]:
# Control panels

    # product drop down menu
product_selector = widgets.Dropdown(
    options=item_lookup[
        'item_id'
    ].tolist(),
    description='Product:',
    layout=widgets.Layout(
        width='360px'
    )
)

    # product history selector
history_selector = widgets.Dropdown(
    options=[
        ('90 days', 90),
        ('180 days', 180),
        ('365 days', 365),
        ('730 days', 730)
    ],
    value=180,
    description='History:',
    layout=widgets.Layout(
        width='260px'
    )
)

    # Text input for current stock
current_stock_input = (
    widgets.BoundedIntText(
        value=0,
        min=0,
        max=100000,
        step=1,
        description='Current stock:',
        layout=widgets.Layout(
            width='260px'
        )
    )
)

    # Slider for stock safety
safety_stock_input = widgets.IntSlider(
    value=15,
    min=0,
    max=50,
    step=5,
    description='Safety stock:',
    continuous_update=False,
    readout_format='d',
    layout=widgets.Layout(
        width='380px'
    )
)

show_actual_input = widgets.Checkbox(
    value=False,
    description='Show actual test demand',
    indent=False
)

update_button = widgets.Button(
    description='Update dashboard',
    button_style='primary',
    icon='refresh',
    layout=widgets.Layout(
        width='220px'
    )
)

dashboard_output = widgets.Output()

### Small metric-card function

In [8]:
def display_metric_cards(metrics):
    cards = []

    for label, value in metrics:
        cards.append(
            f'''
            <div style="
                flex: 1;
                min-width: 150px;
                padding: 14px;
                margin: 5px;
                border: 1px solid #dddddd;
                border-radius: 8px;
                background: #f8f9fa;
            ">
                <div style="
                    color: #555555;
                    font-size: 13px;
                ">
                    {label}
                </div>

                <div style="
                    font-size: 22px;
                    font-weight: bold;
                    margin-top: 5px;
                ">
                    {value}
                </div>
            </div>
            '''
        )

    display(
        HTML(
            '<div style="display:flex;'
            'flex-wrap:wrap;">'
            + ''.join(cards)
            + '</div>'
        )
    )

In [9]:
# Dashboard update function
def update_dashboard(_=None):
    selected_item = (product_selector.value)

    item_details = (
        item_lookup.loc[ item_lookup['item_id'] == selected_item ]
        .iloc[0]
    )

    selected_forecast = (
        prediction_df.loc[
            prediction_df['item_id']
            == selected_item
        ]
        .sort_values('date')
        .reset_index(drop=True)
    )

    with dashboard_output:
        clear_output(wait=True)

        if len(selected_forecast) != 28:
            display(
                HTML(
                    '<h3 style="color:#b00020;">'
                    'Forecast unavailable</h3>'
                    f'<p>Expected 28 predictions but '
                    f'found {len(selected_forecast)}.'
                    '</p>'
                )
            )
            return

        forecast_start = ( selected_forecast['date'].min())

        selected_history = (feature_df.loc[
                (feature_df['item_id'] == selected_item)
                & (feature_df['date'] < forecast_start)
            ].sort_values('date')
        )

        display_history = (selected_history.tail(history_selector.value))

        recent_history = (selected_history.tail(28))

        recent_total = (recent_history['sales'].sum())

        recent_mean = (recent_history['sales'].mean())

        recent_std = (recent_history['sales'].std(ddof=1))

        if pd.isna(recent_std):
            recent_std = 0

        recent_peak = (recent_history['sales'].max())

        recent_zero_rate = (recent_history['sales'].eq(0).mean())

        recent_p90 = (recent_history['sales'].quantile(0.90))

        forecast_total = (selected_forecast['forecast'].sum())

        forecast_mean = (selected_forecast['forecast'].mean())

        peak_position = (selected_forecast['forecast'].idxmax())

        peak_date = (selected_forecast.loc[peak_position, 'date'])

        peak_forecast = (selected_forecast.loc[peak_position,'forecast'])

        stock_plan = calculate_stock_plan(
            forecast_total=
                forecast_total,
            current_stock=
                current_stock_input.value,
            safety_percentage=
                safety_stock_input.value
        )

        high_demand_threshold = max(
            recent_p90,
            recent_mean + recent_std
        )

        high_average_demand = (
            forecast_mean
            > recent_mean * 1.20
            if recent_mean > 0
            else forecast_mean > 0
        )

        high_peak_demand = (
            peak_forecast
            > high_demand_threshold
        )

        # Application overview
        display(
            HTML(
                '<h2>1. Application overview</h2>'
                '<p>This prototype supports product-level '
                'demand review and indicative stock planning '
                'for the CA_1 store.</p>'
            )
        )

        display_metric_cards([
            ('Store', item_details['store_id']),
            ('Product', item_details['item_id']),
            ('Department', item_details['dept_id']),
            ('Category', item_details['cat_id'])
        ])

        # Demand history
        display(HTML('<h2>2. Demand history</h2>'))

        fig, ax = plt.subplots(
            figsize=(13, 4)
        )

        ax.plot(
            display_history['date'],
            display_history['sales'],
            color='#4C78A8',
            linewidth=1.4
        )

        ax.set_title(
            f'Historical demand: '
            f'{selected_item}'
        )

        ax.set_xlabel('Date')
        ax.set_ylabel('Daily sales')
        ax.grid(alpha=0.25)

        plt.tight_layout()
        plt.show()
        plt.close(fig)

        display_metric_cards([
            (
                'Recent 28-day sales',
                f'{recent_total:,.0f} units'
            ),
            (
                'Daily average',
                f'{recent_mean:,.2f} units'
            ),
            (
                'Recent peak',
                f'{recent_peak:,.0f} units'
            ),
            (
                'Zero-sales days',
                f'{recent_zero_rate:.1%}'
            )
        ])

        # Forecast
        display(
            HTML(
                '<h2>3. Forecast</h2>'
            )
        )

        display_metric_cards([
            (
                'Forecast period',
                '28 days'
            ),
            (
                'Predicted total',
                f'{forecast_total:,.1f} units'
            ),
            (
                'Expected daily demand',
                f'{forecast_mean:,.2f} units'
            ),
            (
                'Peak forecast date',
                peak_date.strftime(
                    '%d %b %Y'
                )
            ),
            (
                'Peak forecast',
                f'{peak_forecast:,.2f} units'
            )
        ])

        fig, ax = plt.subplots(
            figsize=(13, 4)
        )

        ax.plot(
            selected_forecast['date'],
            selected_forecast[
                'forecast'
            ],
            marker='o',
            linewidth=2,
            color='#E15759',
            label='Random Forest forecast'
        )

        if show_actual_input.value:
            ax.plot(
                selected_forecast['date'],
                selected_forecast['sales'],
                marker='o',
                linewidth=1.5,
                color='black',
                label='Actual demand'
            )

        ax.set_title(
            f'28-day demand forecast: '
            f'{selected_item}'
        )

        ax.set_xlabel('Date')
        ax.set_ylabel('Units')
        ax.legend()
        ax.grid(alpha=0.25)

        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Decision support
        display(HTML('<h2>4. Decision support</h2>'))

        display_metric_cards([
            (
                'Current stock',
                f'{current_stock_input.value:,} '
                'units'
            ),
            (
                'Base requirement',
                f'{stock_plan["base_requirement"]:,} '
                'units'
            ),
            (
                'Safety-stock adjustment',
                f'{stock_plan["safety_stock"]:,} '
                'units'
            ),
            (
                'Suggested stock',
                f'{stock_plan["suggested_stock"]:,} '
                'units'
            ),
            (
                'Estimated stock gap',
                f'{stock_plan["stock_gap"]:,} '
                'units'
            )
        ])

        if high_average_demand:
            display(
                HTML(
                    '<div style="padding:12px;'
                    'background:#fff3cd;'
                    'border-radius:6px;">'
                    '<b>High-demand warning:</b> '
                    'expected average demand is '
                    'at least 20% above the recent '
                    'daily average.'
                    '</div>'
                )
            )

        elif high_peak_demand:
            display(
                HTML(
                    '<div style="padding:12px;'
                    'background:#fff3cd;'
                    'border-radius:6px;">'
                    '<b>Peak-demand warning:</b> '
                    'the maximum forecast exceeds '
                    'the recent high-demand threshold.'
                    '</div>'
                )
            )

        else:
            display(
                HTML(
                    '<div style="padding:12px;'
                    'background:#d4edda;'
                    'border-radius:6px;">'
                    'No high-demand warning was '
                    'triggered.'
                    '</div>'
                )
            )

        if stock_plan[ 'stock_gap' ] > 0:
            display(
                HTML(
                    '<p><b>Indicative action:</b> '
                    f'consider obtaining '
                    f'{stock_plan["stock_gap"]:,} '
                    'additional units.</p>'
                )
            )

        else:
            display(
                HTML(
                    '<p><b>Indicative action:</b> '
                    'current stock covers the '
                    'suggested requirement under '
                    'the selected assumptions.</p>'
                )
            )

        # Model performance
        display(
            HTML('<h2>5. Model performance</h2>')
        )

        comparison_models = [
            'random_forest_tuned',
            'rolling_mean_28',
            'seasonal_naive_7'
        ]

        performance_table = (
            metrics_df.loc[
                metrics_df['model'].isin(
                    comparison_models
                ),
                [ 'model', 'MAE', 'RMSE', 'WAPE']
            ]
            .copy()
        )

        performance_table[
            'model'
        ] = performance_table[
            'model'
        ].replace({
            'random_forest_tuned':
                'Random Forest',
            'rolling_mean_28':
                '28-day rolling mean',
            'seasonal_naive_7':
                'Seasonal naïve'
        })

        performance_table[
            'WAPE (%)'
        ] = ( performance_table['WAPE'] * 100 )

        performance_table = (
            performance_table[
                ['model', 'MAE', 'RMSE', 'WAPE (%)']
            ]
            .sort_values('WAPE (%)')
            .rename( columns={'model': 'Model'})
        )

        display(
            performance_table.style.format({
                'MAE': '{:.3f}',
                'RMSE': '{:.3f}',
                'WAPE (%)': '{:.2f}'
            })
        )

        display(
            HTML(
                '<p>The Random Forest achieved '
                'the strongest overall final-test '
                'performance, although its advantage '
                'over the 28-day rolling mean was '
                'small.</p>'
            )
        )

        # Explainability
        display(HTML('<h2>6. Model explanation</h2>')
        )

        if importance_df.empty:
            display(
                HTML(
                    '<p>Feature importance was not '
                    'found. Run Notebook 10 first.</p>'
                )
            )

        else:
            top_features = (
                importance_df
                .sort_values(
                    importance_value,
                    ascending=False
                )
                .head(8)
                .sort_values(
                    importance_value
                )
            )

            fig, ax = plt.subplots(figsize=(10, 5))

            ax.barh(
                top_features[
                    'original_feature'
                ],
                top_features[
                    importance_value
                ],
                color='#59A14F'
            )

            ax.set_title('Most influential model predictors')

            ax.set_xlabel(importance_label)

            ax.grid(
                axis='x',
                alpha=0.25
            )

            plt.tight_layout()
            plt.show()
            plt.close(fig)

            display(
                HTML(
                    '<p>These are global explanations '
                    'for the complete Random Forest. '
                    'They are not causal effects and '
                    'are not specific explanations for '
                    'only the selected product.</p>'
                )
            )

        # Reliability
        display(
            HTML(
                '''
                <h2>7. Reliability and limitations</h2>

                <div style="
                    padding: 14px;
                    background: #f8d7da;
                    border-radius: 6px;
                ">
                    <b>Uncertainty warning:</b>
                    forecasts are estimates rather
                    than guaranteed sales quantities.
                    This prototype provides point
                    forecasts and does not calculate
                    formal prediction intervals.
                </div>

                <ul>
                    <li>
                        The experiment covers one store,
                        CA_1, and 108 selected products.
                    </li>
                    <li>
                        Product-level sales are
                        intermittent and frequently zero.
                    </li>
                    <li>
                        Unexpected promotions, stockouts
                        or external events may affect
                        actual demand.
                    </li>
                    <li>
                        The safety-stock percentage is a
                        user-selected planning assumption,
                        not an optimised inventory policy.
                    </li>
                    <li>
                        The M5 data does not contain
                        complete inventory, replenishment,
                        lead-time or service-level data.
                    </li>
                    <li>
                        Recommendations are advisory and
                        should be reviewed by a human
                        decision-maker.
                    </li>
                </ul>
                '''
            )
        )

### Dashboard update function

In [13]:
def update_dashboard(_=None):
    selected_item = (product_selector.value)

    item_details = ( item_lookup.loc[item_lookup['item_id'] == selected_item].iloc[0] )

    selected_forecast = ( prediction_df.loc[prediction_df['item_id'] == selected_item]
        .sort_values('date')
        .reset_index(drop=True)
    )

    with dashboard_output:
        clear_output(wait=True)

        if len(selected_forecast) != 28:
            display(
                HTML(
                    '<h3 style="color:#b00020;">'
                    'Forecast unavailable</h3>'
                    f'<p>Expected 28 predictions but '
                    f'found {len(selected_forecast)}.'
                    '</p>'
                )
            )
            return

        forecast_start = (selected_forecast[ 'date' ].min())

        selected_history = (
            feature_df.loc[
                (feature_df['item_id'] == selected_item) & (feature_df['date'] < forecast_start)
            ].sort_values('date')
        )

        display_history = (selected_history.tail(history_selector.value))

        recent_history = (selected_history.tail(28))

        recent_total = (recent_history['sales'].sum())

        recent_mean = (recent_history['sales'].mean())

        recent_std = (recent_history['sales'].std(ddof=1))

        if pd.isna(recent_std):
            recent_std = 0

        recent_peak = (recent_history['sales'].max())

        recent_zero_rate = (
            recent_history['sales']
            .eq(0)
            .mean()
        )

        recent_p90 = (recent_history['sales'].quantile(0.90))

        forecast_total = (selected_forecast['forecast'].sum())

        forecast_mean = (selected_forecast['forecast'].mean())

        peak_position = (selected_forecast['forecast'].idxmax())

        peak_date = (selected_forecast.loc[ peak_position, 'date' ])

        peak_forecast = (selected_forecast.loc[peak_position, 'forecast'])

        stock_plan = calculate_stock_plan(
            forecast_total=
                forecast_total,
            current_stock=
                current_stock_input.value,
            safety_percentage=
                safety_stock_input.value
        )

        high_demand_threshold = max(
            recent_p90,
            recent_mean + recent_std
        )

        high_average_demand = (
            forecast_mean
            > recent_mean * 1.20
            if recent_mean > 0
            else forecast_mean > 0
        )

        high_peak_demand = (
            peak_forecast
            > high_demand_threshold
        )

        # Application overview
        display(
            HTML(
                '<h2>1. Application overview</h2>'
                '<p>This prototype supports product-level '
                'demand review and indicative stock planning '
                'for the CA_1 store.</p>'
            )
        )

        display_metric_cards([
            (
                'Store',
                item_details['store_id']
            ),
            (
                'Product',
                item_details['item_id']
            ),
            (
                'Department',
                item_details['dept_id']
            ),
            (
                'Category',
                item_details['cat_id']
            )
        ])

        # Demand history
        display(
            HTML('<h2>2. Demand history</h2>')
        )

        fig, ax = plt.subplots(figsize=(13, 4))

        ax.plot(
            display_history['date'],
            display_history['sales'],
            color='#4C78A8',
            linewidth=1.4
        )

        ax.set_title(
            f'Historical demand: '
            f'{selected_item}'
        )

        ax.set_xlabel('Date')
        ax.set_ylabel('Daily sales')
        ax.grid(alpha=0.25)

        plt.tight_layout()
        plt.show()
        plt.close(fig)

        display_metric_cards([
            (
                'Recent 28-day sales',
                f'{recent_total:,.0f} units'
            ),
            (
                'Daily average',
                f'{recent_mean:,.2f} units'
            ),
            (
                'Recent peak',
                f'{recent_peak:,.0f} units'
            ),
            (
                'Zero-sales days',
                f'{recent_zero_rate:.1%}'
            )
        ])

        # Forecast
        display(
            HTML('<h2>3. Forecast</h2>')
        )

        display_metric_cards([
            (
                'Forecast period',
                '28 days'
            ),
            (
                'Predicted total',
                f'{forecast_total:,.1f} units'
            ),
            (
                'Expected daily demand',
                f'{forecast_mean:,.2f} units'
            ),
            (
                'Peak forecast date',
                peak_date.strftime(
                    '%d %b %Y'
                )
            ),
            (
                'Peak forecast',
                f'{peak_forecast:,.2f} units'
            )
        ])

        fig, ax = plt.subplots(figsize=(13, 4))

        ax.plot(
            selected_forecast['date'],
            selected_forecast[
                'forecast'
            ],
            marker='o',
            linewidth=2,
            color='#E15759',
            label='Random Forest forecast'
        )

        if show_actual_input.value:
            ax.plot(
                selected_forecast['date'],
                selected_forecast['sales'],
                marker='o',
                linewidth=1.5,
                color='black',
                label='Actual demand'
            )

        ax.set_title(
            f'28-day demand forecast: '
            f'{selected_item}'
        )

        ax.set_xlabel('Date')
        ax.set_ylabel('Units')
        ax.legend()
        ax.grid(alpha=0.25)

        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Decision support
        display(
            HTML('<h2>4. Decision support</h2>')
        )

        display_metric_cards([
            (
                'Current stock',
                f'{current_stock_input.value:,} '
                'units'
            ),
            (
                'Base requirement',
                f'{stock_plan["base_requirement"]:,} '
                'units'
            ),
            (
                'Safety-stock adjustment',
                f'{stock_plan["safety_stock"]:,} '
                'units'
            ),
            (
                'Suggested stock',
                f'{stock_plan["suggested_stock"]:,} '
                'units'
            ),
            (
                'Estimated stock gap',
                f'{stock_plan["stock_gap"]:,} '
                'units'
            )
        ])

        if high_average_demand:
            display(
                HTML(
                    '<div style="padding:12px;'
                    'background:#fff3cd;'
                    'border-radius:6px;">'
                    '<b>High-demand warning:</b> '
                    'expected average demand is '
                    'at least 20% above the recent '
                    'daily average.'
                    '</div>'
                )
            )

        elif high_peak_demand:
            display(
                HTML(
                    '<div style="padding:12px;'
                    'background:#fff3cd;'
                    'border-radius:6px;">'
                    '<b>Peak-demand warning:</b> '
                    'the maximum forecast exceeds '
                    'the recent high-demand threshold.'
                    '</div>'
                )
            )

        else:
            display(
                HTML(
                    '<div style="padding:12px;'
                    'background:#d4edda;'
                    'border-radius:6px;">'
                    'No high-demand warning was '
                    'triggered.'
                    '</div>'
                )
            )

        if stock_plan[
            'stock_gap'
        ] > 0:
            display(
                HTML(
                    '<p><b>Indicative action:</b> '
                    f'consider obtaining '
                    f'{stock_plan["stock_gap"]:,} '
                    'additional units.</p>'
                )
            )

        else:
            display(
                HTML(
                    '<p><b>Indicative action:</b> '
                    'current stock covers the '
                    'suggested requirement under '
                    'the selected assumptions.</p>'
                )
            )

        # Model performance
        display(
            HTML('<h2>5. Model performance</h2>')
        )

        comparison_models = [
            'random_forest_tuned',
            'rolling_mean_28',
            'seasonal_naive_7'
        ]

        performance_table = (
            metrics_df.loc[
                metrics_df['model'].isin(
                    comparison_models
                ),
                [ 'model', 'MAE', 'RMSE', 'WAPE' ]
            ]
            .copy()
        )

        performance_table[
            'model'
        ] = performance_table[
            'model'
        ].replace({
            'random_forest_tuned':
                'Random Forest',
            'rolling_mean_28':
                '28-day rolling mean',
            'seasonal_naive_7':
                'Seasonal naïve'
        })

        performance_table[
            'WAPE (%)'
        ] = (
            performance_table['WAPE']
            * 100
        )

        performance_table = (
            performance_table[
                [ 'model', 'MAE', 'RMSE', 'WAPE (%)' ]
            ]
            .sort_values('WAPE (%)')
            .rename(
                columns={
                    'model': 'Model'
                }
            )
        )

        display(
            performance_table.style.format({
                'MAE': '{:.3f}',
                'RMSE': '{:.3f}',
                'WAPE (%)': '{:.2f}'
            })
        )

        display(
            HTML(
                '<p>The Random Forest achieved '
                'the strongest overall final-test '
                'performance, although its advantage '
                'over the 28-day rolling mean was '
                'small.</p>'
            )
        )

        # Explainability
        display(
            HTML('<h2>6. Model explanation</h2>')
        )

        if importance_df.empty:
            display(
                HTML(
                    '<p>Feature importance was not '
                    'found. Run Notebook 10 first.</p>'
                )
            )

        else:
            top_features = (
                importance_df
                .sort_values(
                    importance_value,
                    ascending=False
                )
                .head(8)
                .sort_values(
                    importance_value
                )
            )

            fig, ax = plt.subplots(figsize=(10, 5))

            ax.barh(
                top_features[
                    'original_feature'
                ],
                top_features[
                    importance_value
                ],
                color='#59A14F'
            )

            ax.set_title('Most influential model predictors')

            ax.set_xlabel(importance_label)

            ax.grid(
                axis='x',
                alpha=0.25
            )

            plt.tight_layout()
            plt.show()
            plt.close(fig)

            display(
                HTML(
                    '<p>These are global explanations '
                    'for the complete Random Forest. '
                    'They are not causal effects and '
                    'are not specific explanations for '
                    'only the selected product.</p>'
                )
            )


In [14]:
# +RUNNING the display
update_button.on_click(update_dashboard)

application_controls = widgets.VBox([
    widgets.HTML('''
        <h1>
            Retail Demand Decision-Support Prototype
        </h1>

        <p>
            Select a product and enter the planning
            assumptions, then update the dashboard.
        </p>
        '''),

    widgets.HBox([
        product_selector,
        history_selector
    ]),

    widgets.HBox([
        current_stock_input,
        safety_stock_input
    ]),

    show_actual_input,
    update_button
])

display(application_controls, dashboard_output)

update_dashboard()

Output()